# Task: Fruit Ripeness Classification (Color Space Comparison)
In this experiment, we will compare how **RGB/Grayscale**, **HSV**, and **CIELAB** color spaces represent different stages of fruit ripeness (unripe, ripe, overripe).

**Goal**: We are not building a full machine learning classifier here. Instead, we are exploring which color space provides the best "features" to distinguish between the three classes.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

def show_three(img1, img2, img3, title1="Unripe", title2="Ripe", title3="Overripe", cmap=None):
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    for ax, img, title in zip(axs, [img1, img2, img3], [title1, title2, title3]):
        if cmap:
            ax.imshow(img, cmap=cmap)
        else:
            ax.imshow(img)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

### 1. Load Original Images
Let's load the dataset and convert BGR to RGB so Matplotlib can display them correctly.

In [ ]:
# Load images
unripe = cv2.imread('../assets/banana_unripe.jpg')
ripe = cv2.imread('../assets/banana_ripe.jpg')
overripe = cv2.imread('../assets/banana_overripe.jpg')

# Check if images loaded successfully
if unripe is None or ripe is None or overripe is None:
    print("Error: Could not load one or more images. Please check the paths.")
else:
    # Convert to RGB
    unripe_rgb = cv2.cvtColor(unripe, cv2.COLOR_BGR2RGB)
    ripe_rgb = cv2.cvtColor(ripe, cv2.COLOR_BGR2RGB)
    overripe_rgb = cv2.cvtColor(overripe, cv2.COLOR_BGR2RGB)

    show_three(unripe_rgb, ripe_rgb, overripe_rgb, "Original: Unripe (Green)", "Original: Ripe (Yellow)", "Original: Overripe (Spots)")

### 2. Flow 1: RGB -> Grayscale -> Binary Thresholding
Let's see if Grayscale intensity can distinguish the ripeness. We will apply an inverted binary threshold to detect "dark spots".

In [ ]:
if unripe is not None:
    # Convert to Grayscale
    unripe_gray = cv2.cvtColor(unripe, cv2.COLOR_BGR2GRAY)
    ripe_gray = cv2.cvtColor(ripe, cv2.COLOR_BGR2GRAY)
    overripe_gray = cv2.cvtColor(overripe, cv2.COLOR_BGR2GRAY)

    show_three(unripe_gray, ripe_gray, overripe_gray, "Grayscale Unripe", "Grayscale Ripe", "Grayscale Overripe", cmap='gray')

    # Binary Thresholding: pixels darker than 100 become white (255), rest become black (0)
    _, thresh_unripe = cv2.threshold(unripe_gray, 100, 255, cv2.THRESH_BINARY_INV)
    _, thresh_ripe = cv2.threshold(ripe_gray, 100, 255, cv2.THRESH_BINARY_INV)
    _, thresh_overripe = cv2.threshold(overripe_gray, 100, 255, cv2.THRESH_BINARY_INV)

    show_three(thresh_unripe, thresh_ripe, thresh_overripe, "Threshold (Spots) Unripe", "Threshold (Spots) Ripe", "Threshold (Spots) Overripe", cmap='gray')

**Observation from Grayscale/Binary**: 
Notice how it successfully isolates the dark spots in the overripe banana! However, the grayscale intensity of the Unripe (green) and Ripe (yellow) bananas are very similar. It fails to confidently distinguish between green and yellow.

### 3. Flow 2: RGB -> HSV (Hue, Saturation, Value)
HSV separates color (Hue) from lighting. Let's look at the **Hue** channel specifically.

In [ ]:
if unripe is not None:
    # Convert to HSV
    unripe_hsv = cv2.cvtColor(unripe, cv2.COLOR_BGR2HSV)
    ripe_hsv = cv2.cvtColor(ripe, cv2.COLOR_BGR2HSV)
    overripe_hsv = cv2.cvtColor(overripe, cv2.COLOR_BGR2HSV)

    # Extract Hue channel (Index 0)
    h_unripe = unripe_hsv[:,:,0]
    h_ripe = ripe_hsv[:,:,0]
    h_overripe = overripe_hsv[:,:,0]

    # Display Hue channel. We use the 'hsv' colormap to make it intuitive.
    show_three(h_unripe, h_ripe, h_overripe, "Hue Channel (Unripe)", "Hue Channel (Ripe)", "Hue Channel (Overripe)", cmap='hsv')

**Observation from HSV**: 
The Hue channel clearly shows a difference in the dominant color. The Unripe banana has pixel values falling in the green spectrum, whereas the Ripe banana shifts towards the yellow spectrum. The Value (V) channel (not shown above) would mimic the Grayscale results for finding dark spots.

### 4. Flow 3: RGB -> CIELAB (L*a*b*)
CIELAB mimics human vision. 
- **L***: Lightness
- **a***: Green (negative) to Red (positive)
- **b***: Blue (negative) to Yellow (positive)

In [ ]:
if unripe is not None:
    # Convert to LAB
    unripe_lab = cv2.cvtColor(unripe, cv2.COLOR_BGR2LAB)
    ripe_lab = cv2.cvtColor(ripe, cv2.COLOR_BGR2LAB)
    overripe_lab = cv2.cvtColor(overripe, cv2.COLOR_BGR2LAB)

    # Extract 'a' channel (Index 1) - Green to Red axis
    # Note: OpenCV shifts the range to 0-255. 
    # Values < 128 are towards Green. Values > 128 are towards Red.
    a_unripe = unripe_lab[:,:,1]
    a_ripe = ripe_lab[:,:,1]
    a_overripe = overripe_lab[:,:,1]

    show_three(a_unripe, a_ripe, a_overripe, "LAB 'a' Channel (Unripe)", "LAB 'a' Channel (Ripe)", "LAB 'a' Channel (Overripe)", cmap='gray')
    
    # Extract 'b' channel (Index 2) - Blue to Yellow axis
    # Values > 128 are towards Yellow.
    b_unripe = unripe_lab[:,:,2]
    b_ripe = ripe_lab[:,:,2]
    b_overripe = overripe_lab[:,:,2]
    
    show_three(b_unripe, b_ripe, b_overripe, "LAB 'b' Channel (Unripe)", "LAB 'b' Channel (Ripe)", "LAB 'b' Channel (Overripe)", cmap='gray')

**Observation from CIELAB**:
- In the **'a' channel** (Green-Red), the Unripe banana appears distinctly darker (more negative/closer to 0 in OpenCV's scale) because it is highly green. The Ripe banana is much lighter.
- In the **'b' channel** (Blue-Yellow), the Ripe banana is very bright (highly yellow), while the brown spots in the Overripe banana show reduced yellow intensity.

### Conclusion:
If you want to build a classifier for fruit ripeness:
1. **CIELAB 'a' channel** or **HSV 'Hue'** is perfect for detecting if the fruit is **Unripe (Green)**.
2. **Grayscale Thresholding** or **HSV 'Value'** is perfect for detecting if the fruit is **Overripe (Brown/Black Spots)**.
3. By combining these simple rules, you can create a highly accurate ripeness classifier without even needing Machine Learning!